# Before Usage Tests

Run this notebook after programming or updating the overlay, before loading experiment waveforms. It checks RFDC status and verifies that both DAC BRAM players can be written and read back through their MMIO arrays.

## Imports and Overlay Load

Instantiating `OverlayController` configures the RFSoC clocks, downloads the bitstream, binds `dac0` and `dac2`, and disables both DAC players.

In [ ]:
import time
import numpy as np
from firmware import OverlayController
ol = OverlayController("overlays/rfsocawg.bit")
info = ol.info()
info

In [ ]:
BUF_LEN = int(info["dac0"]["bram_int16_samples"])
DAC0_SR = float(info["rfdc"]["dac0_sampling_rate_gsps"]) * 1e9
DAC2_SR = float(info["rfdc"]["dac2_sampling_rate_gsps"]) * 1e9

BUF_LEN, DAC0_SR, DAC2_SR

## RFDC Diagnostics

These helpers avoid broad `dir(overlay)` probing because some generated PYNQ attributes instantiate incomplete IP drivers when accessed.

In [ ]:
def find_rfdc(overlay):
    candidate_names = ["rfdc", "xrfdc", "usp_rf_data_converter_1", "usp_rf_data_converter"]
    for name in getattr(overlay, "ip_dict", {}):
        if "rfdc" in name.lower() or "rf_data_converter" in name.lower():
            candidate_names.append(name.replace("/", "_"))

    seen = set()
    errors = []
    for name in candidate_names:
        if name in seen:
            continue
        seen.add(name)
        try:
            obj = getattr(overlay, name)
        except Exception as e:
            errors.append((name, repr(e)))
            continue
        try:
            getattr(obj, "IPStatus")
            getattr(obj, "dac_tiles")
            getattr(obj, "adc_tiles")
        except Exception as e:
            errors.append((name, repr(e)))
            continue
        print(f"RFDC object: {name}")
        return obj

    print("RFDC candidate access errors:", errors)
    raise RuntimeError(f"RFDC object not found. Candidates: {candidate_names}")


def print_rfdc_summary():
    rfdc = find_rfdc(ol)
    print("IPStatus:", rfdc.IPStatus)
    for kind, tiles in [("DAC", rfdc.dac_tiles), ("ADC", rfdc.adc_tiles)]:
        for tile_id, tile in enumerate(tiles):
            try:
                print(f"{kind} tile {tile_id}: PLLLockStatus={tile.PLLLockStatus}, FIFOStatus={tile.FIFOStatus}")
            except Exception as e:
                print(f"{kind} tile {tile_id}: tile status read failed:", e)
            for block_id, block in enumerate(tile.blocks):
                try:
                    st = block.BlockStatus
                    print(
                        f"  block {block_id}: SamplingFreq={st.get('SamplingFreq')}, "
                        f"DigitalPathEnabled={st.get('DigitalPathEnabled')}, "
                        f"DataPathClocksStatus={st.get('DataPathClocksStatus')}"
                    )
                except Exception as e:
                    print(f"  block {block_id}: block status read failed:", e)

In [ ]:
print_rfdc_summary()

## BRAM MMIO Readback Test

The test disables the DAC player, writes deterministic `int16` patterns into BRAM through the player buffer, and reads the same memory back. It does not enable RF output.

In [ ]:
def _as_i16_pattern(pattern, n):
    if pattern == "zeros":
        return np.zeros(n, dtype=np.int16)
    if pattern == "ones":
        return np.full(n, 0x7fff, dtype=np.int16)
    if pattern == "alternating":
        x = np.empty(n, dtype=np.int16)
        x[0::2] = np.int16(0x5555)
        x[1::2] = np.int16(-0x5556)  # 0xAAAA as signed int16
        return x
    if pattern == "ramp":
        return (np.arange(n, dtype=np.int32) & 0xffff).astype(np.int16)
    if pattern == "prbs":
        rng = np.random.default_rng(0x12288)
        return rng.integers(-32768, 32767, size=n, dtype=np.int16)
    raise ValueError(pattern)


def streamer_enable(player, enabled):
    if enabled:
        player.enable()
    else:
        player.disable()


def bram_array_test(player, n=None, patterns=("zeros", "ones", "alternating", "ramp", "prbs")):
    streamer_enable(player, False)
    n = player.capacity if n is None else min(int(n), player.capacity)
    failures = []
    for pattern in patterns:
        expected = _as_i16_pattern(pattern, n)
        player.buffer[:n] = expected
        time.sleep(0.02)
        got = np.array(player.buffer[:n], dtype=np.int16, copy=True)
        bad = np.flatnonzero(got != expected)
        if len(bad):
            i = int(bad[0])
            failures.append((pattern, len(bad), i, int(expected[i]), int(got[i])))
            print(
                f"FAIL {player.name} {pattern}: mismatches={len(bad)}, first={i}, "
                f"expected={int(expected[i])}, got={int(got[i])}"
            )
        else:
            print(f"PASS {player.name} {pattern}: {n} samples")
    if failures:
        raise AssertionError(f"BRAM array test failed: {failures}")
    return True

In [ ]:
test_len = min(BUF_LEN, 2**18)
bram_array_test(ol.dac0, n=test_len)
bram_array_test(ol.dac2, n=test_len)